## 1. Data Preprocessing
In this section, we import the raw ESS data and prepare it for analysis by handling missing values and transforming variables.

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn.external.docscrape import header

# Load Dataset
zip_path = "ESS9e03_3-ESS10e03_3-ESS10SCe03_2-ESS11e04_1-subset.zip"
csv_inside = "ESS9e03_3-ESS10e03_3-ESS10SCe03_2-ESS11e04_1-subset.csv"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open(csv_inside) as f:
        df = pd.read_csv(f, low_memory=False)

print("CSV Load Successful!")
print(f"Dataset Size: {df.shape[0]} rows x {df.shape[1]} columns")

CSV Load Successful!
Dataset Size: 159320 rows x 635 columns


### Data Exploration
The original ESS dataset contains hundreds of variables. For this study, we have isolated 34 specific variables categorized into:

Demographics: yrbrn, agea, gndr, etc.

Schwartz Human Values: The 21-item battery (e.g., ipcrtiv to ipadvnt).

Socio-Political Attitudes: gincdif (Redistribution) and Institutional Trust (trstplt).

### Data Cleansing
To ensure the data is research-ready, we perform the following:
1. **Checked for duplicates**
2. **Checked Missing Values:** Convert ESS codes (7, 8, 9, 77, 88, 99) to `NaN`.
3. **Scale Reversal:** We transform the 1- 6 scale to ensure **6 = Strong Agreement**.


#### 1. Checked for duplicates

All value variables (ipcrtiv, imprich, impfun, ...) have 22074 NaN

This usually happens in ESS when you merged rounds OR loaded multiple rounds in one file
because Schwartz values are NOT asked in every round.

So this is normal, not an error.

#### 3. MRAT (Mean Rating) Correction.

1. calculate the average score each person gave across all 21 original items. 

In [ ]:
print([c for c in df.columns if 'env' in c])

2. Apply the MRAT Correction (Centering)

In [ ]:
# 1.Define the specific columns we want to keep for the final analysis
centered_columns = [f'{v}_centered' for v in schwartz_values]
normalized_columns = [f'{v}_normalized' for v in schwartz_values]
analysis_columns = [
    'idno', 'cntry', 'essround', 'generation', 'yrbrn', 'agea', 'gndr', 'anweight', 'lrscale', 'stflife', 'trstprl', 'gincdif', 'mrat_score'
    ] + centered_columns + normalized_columns

# create final analysis file
df[analysis_columns].to_csv("ess_final_analysis.csv", index=False)
print("✅ Master Analysis File Saved: 'ess_final_analysis.csv'")

#print(centered_columns)
#print(df[normalized_columns].head())
#print(df[analysis_columns].head())


## 2. Descriptive Analysis

### 1. Sample Composition (Demographics)
We would like to show the distribution of the respondents to tell how many people from each generation are represented.

In [ ]:
# Final check: Average Universalism by Generation
print("### Mean Universalism Score by Generation")
display(df.groupby('generation')['Universalism'].mean().sort_values(ascending=False))